# 02. Stokastisk build 5: Kaggle 2×T4 validation (Mode A standalone / Mode B OCI-assisted)

**Accelerator: GPU T4 x2** · fresh Kaggle session · Python 3.13 · generated by `scripts/build_kaggle_diagnostic_notebooks.py`

Immutable `stokastisk-0.1.0-5` only (size + SHA-256 verified before install). Qwen/Qwen2.5-0.5B-Instruct, FP16, TRITON_ATTN, TP=1, PP=1, 2048 tokens, gpu_memory_utilization 0.60. **No Enigma runtime connection and no runtime token in either mode.** Choose the mode in the first code cell: `standalone` (default; no secret needed) or `oci-assisted` (needs the Kaggle Secret `ENIGMA_DIAG_KEY`; stops before any GPU work if the OCI diagnostic route does not acknowledge).

**Kaggle Secrets (Add-ons → Secrets; tick each one for this notebook).** `ENIGMA_DIAG_KEY` = the 56-character diagnostic key (`edx_…`, issued once by `scripts/diag_keys.py create` on the OCI host); it authenticates only `POST /api/v1/diag/events`. `ENIGMA_RUNTIME_TOKEN` = the 60-character Enigma runtime key (`enig_rt_…`, issued in the Enigma web app); it authenticates only runtime traffic. They are never interchangeable; no cloud-provider credential (OCI, AWS) is ever needed in Kaggle. If a credential preflight cell reports a problem, fix the secret in the sidebar and **re-run only that cell** (and the cell after it): the attempt id, event sequence and earlier verdicts are kept. A new Kaggle session always starts from the top.

**Evidence durability.** Every step prints flushed `MARK` and `VERDICT` lines and appends them to an fsynced `markers.jsonl` in a new, never-reused attempt directory. After a complete platform termination the VM's files may be gone (on 2026-11-26 a new session had to re-clone everything), but cell output printed before the failure was kept in the saved notebook. **Missing files do not prove that no crash occurred.** After a fatal session, copy the last `MARK` lines (UTC time, `t` = seconds since the prelude) into the incident report.

Run the cells top to bottom (or *Run all*). Every step ends with `VERDICT <step>: PASS | FAIL | INCONCLUSIVE`; anything but PASS raises and stops the run. Then run the final **evidence export** cell manually.

In [ ]:
# P0. Prelude: environment guard, attempt id, markers and verdicts. Standard library only: no network, no GPU library is imported.
import os, sys, json, time, uuid, platform, shutil, subprocess, re, pathlib
NOTEBOOK = "02_stokastisk_build5_kaggle_t4_validation"
EXPECT = "gpu"                                   # "cpu" = Accelerator None; "gpu" = GPU T4 x2
assert os.path.isdir("/kaggle/working"), "Kaggle only: refusing to run anywhere else (never on a developer laptop)"


class DiagnosticStop(Exception):
    """Raised by every non-PASS verdict so that 'Run all' stops here. Later cells must never run after a failure."""


HAS_GPU_DEVICE = os.path.exists("/dev/nvidia0")
if EXPECT == "cpu" and HAS_GPU_DEVICE:
    raise DiagnosticStop("INCONCLUSIVE: wrong environment. A GPU is attached; set Accelerator = None and start a NEW session")
if EXPECT == "gpu" and not HAS_GPU_DEVICE:
    raise DiagnosticStop("INCONCLUSIVE: wrong environment. No GPU; set Accelerator = GPU T4 x2 and start a NEW session")
assert sys.version_info[:2] == (3, 13), f"expected Python 3.13 (build-5 wheel is cp313), got {platform.python_version()}"
REQUIRE_NO_INPUTS = False                       # notebook 01 must run with NO attached dataset, competition or utility script
_inputs = sorted(os.listdir("/kaggle/input")) if os.path.isdir("/kaggle/input") else []
if REQUIRE_NO_INPUTS and _inputs:
    raise DiagnosticStop(f"INCONCLUSIVE: wrong environment. Inputs are attached {_inputs}; remove every input and start a NEW session")
# ---- configuration: edit ONLY these lines, before running anything; they are fixed for the whole attempt ----
VALIDATION_MODE = "standalone"   # "standalone" = Mode A (no network to Enigma, no secret needed) | "oci-assisted" = Mode B (ENIGMA_DIAG_KEY required)
RUNTIME_TRANSPORT = "none"
# ------------------------------------------------------------------------------------------------------------
assert VALIDATION_MODE in ("standalone", "oci-assisted", "oci-required"), VALIDATION_MODE
assert RUNTIME_TRANSPORT in ("none", "oci"), RUNTIME_TRANSPORT
EXPERIMENT = ("build5" if EXPECT == "gpu" else "no-sdk") + ("" if VALIDATION_MODE == "oci-required" else "-" + VALIDATION_MODE) \
    + ("" if RUNTIME_TRANSPORT == "none" else "-via-" + RUNTIME_TRANSPORT)   # in every MARK

ATTEMPT = time.strftime("%Y%m%dT%H%M%SZ", time.gmtime()) + "-" + uuid.uuid4().hex[:8]
ATTEMPT_DIR = f"/kaggle/working/enigma-diag-attempts/{NOTEBOOK}/{ATTEMPT}"
os.makedirs(ATTEMPT_DIR, exist_ok=False)               # a new directory per attempt: earlier evidence is never overwritten
T0 = time.monotonic()
RESULTS = {}                                           # verdicts of THIS kernel session only (stale outputs can never satisfy a check)

# Evidence redaction (shared; generated from scripts/kaggle_cells/evidence.py). Defence in depth: credentials are never recorded on purpose, and every
# MARK line and every exported text file is still scrubbed of anything shaped like an Enigma runtime key, a diagnostic key, an AWS key id, a bearer
# token or a JWT before it is written or archived.
import re, pathlib, zipfile
_SECRET_RE = re.compile(r"(?:enig_rt_|edx_)[a-z0-9]{8}_[A-Za-z0-9_-]{10,}|\b(?:AKIA|ASIA)[0-9A-Z]{16}\b|(?i:bearer)\s+[A-Za-z0-9._~+/=-]{8,}"
                        r"|\beyJ[A-Za-z0-9_-]{6,}\.[A-Za-z0-9_-]{6,}\.[A-Za-z0-9_-]*")
_EXPORT_SKIP_DIRS = ("enigma-src",)                    # the public connector clone is not evidence
_EXPORT_SKIP_SUFFIXES = (".whl",)                      # the public 38 MB wheel is not evidence (its sha256 is in the markers)


def redact(text):
    return _SECRET_RE.sub("<REDACTED>", text)


def write_redacted_archive(src_dir, zip_path):
    """Zip src_dir, scrubbing every UTF-8 text file. Returns (files, redactions)."""
    files = hits = 0
    with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
        for p in sorted(pathlib.Path(src_dir).rglob("*")):
            rel = p.relative_to(src_dir)
            if not p.is_file() or p.is_symlink() or rel.parts[0] in _EXPORT_SKIP_DIRS or p.suffix in _EXPORT_SKIP_SUFFIXES:
                continue
            data = p.read_bytes()
            try:
                text = data.decode("utf-8")
            except UnicodeDecodeError:
                z.writestr(str(rel), data)
            else:
                hits += len(_SECRET_RE.findall(text))
                z.writestr(str(rel), redact(text))
            files += 1
    return files, hits


def _read(path):
    try:
        with open(path) as fh:
            return fh.read(65536)
    except OSError:
        return None


def host():
    """Host memory, cgroup v2 memory (current, max, events), shared memory, disk, uptime and load. No environment variables are read."""
    h = {}
    for line in (_read("/proc/meminfo") or "").splitlines():
        k, _, v = line.partition(":")
        if k in ("MemTotal", "MemAvailable", "Shmem"):
            h[k.lower() + "_mib"] = round(int(v.split()[0]) / 1024, 1)
    mib = lambda s: None if s is None or not s.strip().isdigit() else round(int(s) / 2**20, 1)   # noqa: E731 ("max" = unlimited -> None)
    h["cgroup_current_mib"], h["cgroup_max_mib"] = mib(_read("/sys/fs/cgroup/memory.current")), mib(_read("/sys/fs/cgroup/memory.max"))
    h["cgroup_events"] = {k: int(v) for k, _, v in (l.partition(" ") for l in (_read("/sys/fs/cgroup/memory.events") or "").splitlines()) if v.strip().isdigit()}
    for d in ("/kaggle/working", "/tmp", "/dev/shm"):
        try:
            u = shutil.disk_usage(d)
            h["disk_" + d.strip("/").replace("/", "_")] = {"used_gib": round(u.used / 2**30, 2), "free_gib": round(u.free / 2**30, 2)}
        except OSError:
            pass
    up = _read("/proc/uptime")
    h["vm_uptime_s"] = float(up.split()[0]) if up else None
    h["loadavg"] = os.getloadavg()
    return h


def ident():
    return {"pid": os.getpid(), "ppid": os.getppid(), "pgid": os.getpgid(0), "sid": os.getsid(0)}


def mark(name, **fields):
    rec = {"attempt": ATTEMPT, "nb": NOTEBOOK, "experiment": EXPERIMENT, "mode": VALIDATION_MODE, "transport": RUNTIME_TRANSPORT, "marker": name, "utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
           "t": round(time.monotonic() - T0, 2), **ident(), **host(), **fields}
    line = redact(json.dumps(rec, default=str))
    with open(f"{ATTEMPT_DIR}/markers.jsonl", "a") as fh:
        fh.write(line + "\n"); fh.flush(); os.fsync(fh.fileno())
    print("MARK", line, flush=True)
    return rec


def verdict(step, status, reason, **evidence):
    assert status in ("PASS", "FAIL", "INCONCLUSIVE"), status
    RESULTS[step] = {"status": status, "reason": reason, "attempt": ATTEMPT, **evidence}
    mark("verdict", step=step, status=status, reason=reason, **evidence)
    print(f"VERDICT {step}: {status} | {reason}", flush=True)
    if status != "PASS":
        raise DiagnosticStop(f"{step} {status}: {reason}")


def require(*steps):
    """Every prerequisite must have PASSED in this very session (same attempt id)."""
    bad = [s for s in steps if RESULTS.get(s, {}).get("status") != "PASS" or RESULTS[s].get("attempt") != ATTEMPT]
    if bad:
        raise DiagnosticStop(f"INCONCLUSIVE: prerequisite(s) did not PASS in this session: {bad}. Start a new session and run from the top.")


# Survival observation (shared; generated from scripts/kaggle_cells/observe.py). PASSIVE by default: local, fsynced, printed MARK lines only (no
# network at all), with explicit sleep-begin / sleep-end markers, so the saved notebook output brackets a platform termination to within one tick.
# Remote diagnostic markers (OCI deployment) are OPT-IN at a few explicit offsets (bounded: at most 12 per window), each bracketed by local remote-submit / remote-ack markers.
# A missing cgroup OOM counter is reported as "unavailable", never as 0.
def oom_kill_count():
    ev = host()["cgroup_events"]
    return ev.get("oom_kill") if "oom_kill" in ev else None


def tick_schedule(seconds, every, dense_first_s=0.0, dense_every=2.0):
    """Monotonic offsets (s) at which a passive tick is recorded: every `dense_every` s for the first `dense_first_s` s, then every `every` s."""
    out, t = [], 0.0
    while True:
        step = dense_every if t < dense_first_s else every
        t = round(min(seconds, t + step), 3)
        out.append(t)
        if t >= seconds:
            return out


def observe(step, seconds, every=15.0, dense_first_s=0.0, dense_every=2.0, remote_at=(), sample=None, durable_begin=False):
    """Survival window. remote_at: offsets (s) where ONE bounded diagnostic marker is sent (active sampling); () = fully passive.
    durable_begin: the window starts only after the OCI deployment acknowledged an observation-start marker. PASS = full duration elapsed, no OOM kill observed,
    no failed sample, and every requested remote marker acknowledged."""
    remote_at = sorted(float(x) for x in remote_at)
    assert len(remote_at) <= 12, "remote markers per window are bounded"
    oom0 = oom_kill_count()
    if durable_begin:
        rcheck(step, step + "_obs_begin", stage=step, phase="before", note=f"window {seconds}s remote_at={remote_at}")
    start, problems, sent, n = time.monotonic(), [], [], 0
    mark(step + "_begin", seconds=seconds, every=every, dense_first_s=dense_first_s, dense_every=dense_every, remote_at=remote_at,
         oom_state="available" if oom0 is not None else "unavailable")
    for target in tick_schedule(seconds, every, dense_first_s, dense_every):
        n += 1
        mark(step + "_sleep_begin", i=n, until_s=target)
        time.sleep(max(0.0, target - (time.monotonic() - start)))
        el = round(time.monotonic() - start, 2)
        mark(step + "_sleep_end", i=n, elapsed_s=el)
        extra = sample() if sample else {}
        if extra.get("problem"):
            problems.append(extra["problem"])
        while remote_at and el + 1e-6 >= remote_at[0]:
            off = remote_at.pop(0)
            mark(step + "_remote_submit", offset_s=off)
            res = rmark(step + "_alive", stage=step, phase="observe", offset_s=off)
            mark(step + "_remote_result", offset_s=off, code=res.get("code"), acknowledged=bool(res))
            sent.append(bool(res))
            if not res and res.get("code") != "REMOTE_DISABLED":
                problems.append(f"remote marker at {off}s not acknowledged ({res.get('code')})")
        if extra:
            mark(step + "_sample", i=n, **extra)
    el, oom1 = time.monotonic() - start, oom_kill_count()
    delta = "unavailable" if oom0 is None or oom1 is None else oom1 - oom0
    ok = el >= seconds - 1.0 and delta in (0, "unavailable") and not problems
    mark(step + "_complete", observed_s=round(el, 2), ticks=n, remote_sent=len(sent), remote_acknowledged=sum(sent))
    verdict(step, "PASS" if ok else "FAIL", f"observed {el:.0f}/{seconds}s, oom_kill delta {delta}, remote {sum(sent)}/{len(sent)} acknowledged, "
            f"problems {problems[:3]}", observed_s=round(el, 1), oom_kill_delta=delta, ticks=n, remote_sent=len(sent), remote_acknowledged=sum(sent))


def export_evidence():
    """Zip this attempt's directory (only it), redacting every text file. Safe to run after a failure."""
    path = f"/kaggle/working/enigma-diag-attempts/{NOTEBOOK}-{ATTEMPT}.zip"
    files, hits = write_redacted_archive(ATTEMPT_DIR, path)
    print("evidence archive:", path, os.path.getsize(path), "bytes,", files, "files, redactions:", hits, flush=True)
    return path


_os = dict(l.split("=", 1) for l in (_read("/etc/os-release") or "").splitlines() if "=" in l)
_kaggle = {k: os.environ[k] for k in ("KAGGLE_KERNEL_RUN_TYPE", "KAGGLE_DOCKER_IMAGE") if k in os.environ}   # allow-list only; never the raw environment
mark("prelude", python=platform.python_version(), expect=EXPECT, gpu_device=HAS_GPU_DEVICE, os=_os.get("PRETTY_NAME", "").strip('"'), kaggle=_kaggle, inputs=_inputs)
print("ATTEMPT", ATTEMPT, "| markers:", ATTEMPT_DIR + "/markers.jsonl", flush=True)

In [ ]:
# GPU observation helpers: nvidia-smi in a SEPARATE process (this kernel never imports torch, pynvml or CUDA), plus /proc process-group states.
def gpu_query():
    q = "index,name,memory.used,memory.total,utilization.gpu,temperature.gpu,power.draw"
    try:
        r = subprocess.run(["nvidia-smi", f"--query-gpu={q}", "--format=csv,noheader,nounits"], capture_output=True, text=True, timeout=30)
        return {"nvidia_smi_rc": r.returncode, "gpus": [dict(zip(q.split(","), (x.strip() for x in l.split(",")))) for l in r.stdout.strip().splitlines()]}
    except Exception as e:
        return {"nvidia_smi_error": type(e).__name__}


def group_procs(pgid):
    """Every process in process group `pgid`: pid, state, rss (MiB), command name. Read from /proc only."""
    page, out = os.sysconf("SC_PAGE_SIZE"), []
    for p in os.listdir("/proc"):
        if not p.isdigit():
            continue
        st = _read(f"/proc/{p}/stat")
        if not st:
            continue
        rest = st.rsplit(")", 1)[1].split()
        if int(rest[2]) == pgid:
            out.append({"pid": int(p), "state": rest[0], "rss_mib": round(int(rest[21]) * page / 2**20, 1), "comm": st[st.find("(") + 1:st.rfind(")")][:24]})
    return out


def inventory():
    import importlib.metadata as md
    norm = lambda n: re.sub(r"[-_.]+", "-", n).lower()                     # noqa: E731
    return {norm(d.metadata["Name"]): d.version for d in md.distributions() if d.metadata["Name"]}

In [ ]:
# C1. Application credentials (Kaggle Secrets) and the diagnostic transport to the OCI deployment. Shared by notebooks 01, 02 and 03 (generated from
# scripts/kaggle_cells/diag_credentials.py; CPU tests exercise this exact text). Standard library only: no cloud SDK, no cloud-provider credential.
#
# Two DIFFERENT application credentials, never interchangeable:
#   ENIGMA_DIAG_KEY       edx_<8>_<43>      56 chars  authenticates ONLY POST /api/v1/diag/events (notebooks 01, 02 oci-assisted, 03)
#   ENIGMA_RUNTIME_TOKEN  enig_rt_<8>_<43>  60 chars  authenticates ONLY Enigma runtime registration / heartbeats / telemetry / jobs (notebook 03)
# Neither is a cloud-provider key. Values are never printed, stored in os.environ, written to evidence, or sent anywhere but their own endpoint; only a
# category, a code and lengths are ever recorded. Only a VALIDATED value is cached; a failed lookup never poisons the cache. After fixing a secret in
# Kaggle's sidebar (Add-ons -> Secrets), call refresh_credentials() (or just re-run the preflight cell): the attempt id, the event sequence, completed
# verdicts and evidence files are all kept. Note: reading a Kaggle Secret is itself a request to Kaggle's own backend (kaggle_secrets talks to the
# Kaggle web service from inside the VM), so no secret is read during an offline observation window.
import re, json, time, socket, ssl, urllib.request, urllib.error, urllib.parse

DIAG_URL = ""
CRED_SPECS = {
    "ENIGMA_DIAG_KEY": {"category": "diagnostic_key", "pattern": r"edx_[a-z0-9]{8}_[A-Za-z0-9_-]{43}", "length": 56, "prefix": "edx_"},
    "ENIGMA_RUNTIME_TOKEN": {"category": "enigma_runtime_token", "pattern": r"enig_rt_[a-z0-9]{8}_[A-Za-z0-9_-]{43}", "length": 60, "prefix": "enig_rt_"},
}
CRED_GUIDANCE = {
    "SECRET_OK": "credential retrieved and validated (value not shown)",
    "SECRET_MISSING": "no Kaggle Secret with this exact name is available: add it in Add-ons -> Secrets, tick it for this notebook, then re-run this cell",
    "SECRET_ACCESS_DENIED": "the secret exists but this notebook may not read it: in Add-ons -> Secrets tick the checkbox next to it, then re-run this cell",
    "SECRET_EMPTY": "the Kaggle Secret is empty: paste the whole value again, then re-run this cell",
    "SECRET_FORMAT_INVALID": "the value does not have the expected format/length: paste exactly the one-line value (no quotes, no NAME=, no spaces), then re-run this cell",
    "SECRET_RETRIEVAL_FAILED": "Kaggle's secret service did not answer: check Internet is ON, then re-run this cell",
}
_WRONG_TYPE = {
    "ENIGMA_DIAG_KEY": ("Wrong credential type. ENIGMA_DIAG_KEY requires the diagnostic key from the local ENIGMA_DIAG_KEY.txt file, not the Enigma "
                        "runtime token."),
    "ENIGMA_RUNTIME_TOKEN": ("Wrong credential type. ENIGMA_RUNTIME_TOKEN requires the Enigma runtime key (enig_rt_...) issued in the Enigma web app "
                             "(Runtimes), not the diagnostic key."),
}
_AWS_ID = re.compile(r"(AKIA|ASIA)[0-9A-Z]{16}")
_CRED_CACHE = {}                                       # name -> validated value ONLY
_DIAG = {"seq": 0, "last": None}                      # the event sequence survives credential refreshes


def _secret_client():
    from kaggle_secrets import UserSecretsClient
    return UserSecretsClient()


def classify_secret(name, value):
    """(code, facts) for a raw secret value. facts never contain the value, a fragment of it, or its key id."""
    spec = CRED_SPECS[name]
    facts = {"credential": name, "expected_category": spec["category"], "expected_length": spec["length"]}
    if value is None:
        return "SECRET_MISSING", facts
    if not isinstance(value, str):
        return "SECRET_FORMAT_INVALID", facts
    v = value.strip()                                 # a trailing newline from copy/paste is tolerated; anything inside is not
    facts.update(actual_length=len(v), embedded_whitespace=any(c.isspace() for c in v), quoted=v[:1] in "'\"" and len(v) > 0,
                 looks_like_assignment=bool(re.match(r"^[A-Za-z_][A-Za-z0-9_]*\s*=", v)))
    if not v:
        return "SECRET_EMPTY", facts
    if re.fullmatch(spec["pattern"], v):
        facts["supplied_category"] = spec["category"]
        return "SECRET_OK", facts
    for other, ospec in CRED_SPECS.items():
        if other != name and (v.startswith(ospec["prefix"]) or re.search(ospec["pattern"], v)):
            facts["supplied_category"] = ospec["category"]
            return "SECRET_WRONG_TYPE", facts
    if _AWS_ID.search(v):
        facts["supplied_category"] = "aws_access_key"
        return "SECRET_WRONG_TYPE", facts
    facts["supplied_category"] = "unknown"
    return "SECRET_FORMAT_INVALID", facts


def _retrieval_code(exc):
    """Classify a kaggle_secrets exception from its class and message WITHOUT recording the message."""
    if isinstance(exc, ImportError):
        return "SECRET_RETRIEVAL_FAILED"
    text = (type(exc).__name__ + " " + str(exc)).lower()
    if "no user secrets exist" in text or "not found" in text or "does not exist" in text:
        return "SECRET_MISSING"
    if any(w in text for w in ("permission", "denied", "unauthori", "forbidden", "not attached", "not authorized", "403", "401")):
        return "SECRET_ACCESS_DENIED"
    return "SECRET_RETRIEVAL_FAILED"


def credential_message(code, name):
    return _WRONG_TYPE[name] if code == "SECRET_WRONG_TYPE" else CRED_GUIDANCE.get(code, code)


def get_credential(name, refresh=False):
    """(value or None, result). Caches only a validated value; any failure clears the cache entry so a later call retries."""
    if refresh:
        _CRED_CACHE.pop(name, None)
    if name in _CRED_CACHE:
        return _CRED_CACHE[name], {"credential": name, "code": "SECRET_OK", "cached": True, "message": CRED_GUIDANCE["SECRET_OK"]}
    try:
        raw = _secret_client().get_secret(name)
    except Exception as exc:                          # noqa: BLE001 - classified; the exception text is never recorded
        _CRED_CACHE.pop(name, None)
        code = _retrieval_code(exc)
        return None, {"credential": name, "code": code, "error_type": type(exc).__name__, "message": credential_message(code, name)}
    code, facts = classify_secret(name, raw)
    if code != "SECRET_OK":
        _CRED_CACHE.pop(name, None)
        return None, {**facts, "code": code, "message": credential_message(code, name)}
    _CRED_CACHE[name] = raw.strip()
    return _CRED_CACHE[name], {**facts, "code": code, "cached": False, "message": CRED_GUIDANCE["SECRET_OK"]}


def refresh_credentials(*names):
    """Forget cached credentials (all, or the named ones) so the next use re-reads Kaggle Secrets. Nothing else is reset."""
    for n in names or list(_CRED_CACHE):
        _CRED_CACHE.pop(n, None)
    return sorted(_CRED_CACHE)


def check_credential(name, step):
    """Preflight verdict for one credential: PASS only for a correctly typed, correctly formatted value."""
    _, res = get_credential(name, refresh=True)
    ev = {k: v for k, v in res.items() if k != "message"}
    if res["code"] == "SECRET_OK":
        verdict(step, "PASS", f"{name}: {res['message']}", **ev)
        return res
    print(f"CREDENTIAL {name}: {res['code']} -> {res['message']}", flush=True)
    verdict(step, "INCONCLUSIVE", f"{name}: {res['code']}", **ev)


# ------------------------------------------------------------------------------------------------------------- transport to the OCI diagnostic route
DIAG_GUIDANCE = {
    "DIAGNOSTIC_PERSISTENCE_VERIFIED": "the OCI deployment acknowledged a durable write (stored or already stored)",
    "DIAGNOSTIC_AUTH_REJECTED": "the diagnostic route rejected ENIGMA_DIAG_KEY: re-check the Kaggle Secret against the key issued by scripts/diag_keys.py; do not retry blindly",
    "DIAGNOSTIC_KEY_EXPIRED_OR_REVOKED": "ENIGMA_DIAG_KEY is correct but expired or revoked: ask for a newly issued diagnostic key",
    "DIAGNOSTIC_QUOTA_EXCEEDED": "this diagnostic key used its whole event quota: ask for a newly issued diagnostic key",
    "DIAGNOSTIC_NETWORK_TIMEOUT": "the diagnostic endpoint did not answer in time",
    "DIAGNOSTIC_DNS_ERROR": "the diagnostic endpoint's host name did not resolve (Internet OFF?)",
    "DIAGNOSTIC_TLS_ERROR": "TLS / certificate verification failed (verification is never disabled)",
    "DIAGNOSTIC_CONNECTION_ERROR": "the connection was refused or reset",
    "DIAGNOSTIC_HTTP_ERROR": "the diagnostic endpoint answered with an HTTP error",
    "DIAGNOSTIC_INVALID_RESPONSE": "the diagnostic endpoint answered with a malformed acknowledgement",
    "DIAGNOSTIC_PERSISTENCE_NOT_ACKNOWLEDGED": "the answer did not acknowledge every event as durably stored",
    "DIAGNOSTIC_CREDENTIAL_UNAVAILABLE": "no valid ENIGMA_DIAG_KEY: nothing was sent",
    "REMOTE_DISABLED": "remote diagnostics are disabled in this mode: nothing was sent",
}
_SERVER_ERRORS = {"unauthorized", "key_expired_or_revoked", "quota_exhausted", "too_large", "bad_json", "bad_events", "event_not_object",
                  "unknown_field", "missing_field", "bad_field", "bad_identity", "method_not_allowed", "unavailable"}
_TRANSIENT_HTTP = {429, 500, 502, 503, 504}
_TRANSIENT_CODES = {"DIAGNOSTIC_NETWORK_TIMEOUT", "DIAGNOSTIC_CONNECTION_ERROR"}
REMOTE_ENABLED = True                                  # notebook 02 standalone mode turns this off: then no secret is read and nothing is sent
_REMOTE_FIELDS = {"stage", "phase", "http", "latency_s", "target", "exc", "exit_code", "signal", "last_ok", "verdict", "note", "gpu_count",
                  "gpu_mem_used_mib", "gpu_util_pct", "wheel_build", "wheel_sha256", "artifact", "offset_s", "child_pid", "child_exit", "outcome"}


class DiagResult(dict):
    """Truthy only when the OCI deployment durably acknowledged every event."""
    def __bool__(self):
        return self.get("code") == "DIAGNOSTIC_PERSISTENCE_VERIFIED"


class _NoRedirect(urllib.request.HTTPRedirectHandler):
    """Never follow a redirect: the Authorization header must never be replayed to another URL or host."""
    def redirect_request(self, req, fp, code, msg, headers, newurl):
        return None


def _build_opener():
    return urllib.request.build_opener(_NoRedirect(), urllib.request.HTTPSHandler(context=ssl.create_default_context()))


_OPENER = _build_opener()


def _diag_url_ok(url):
    u = urllib.parse.urlsplit(url)
    d = urllib.parse.urlsplit(DIAG_URL)
    # Exactly the generator-configured OCI host, https, the one diagnostic path; never another host (the Authorization header goes nowhere else).
    return bool(DIAG_URL) and u.scheme == "https" and bool(u.hostname) and u.hostname == d.hostname and u.port == d.port \
        and u.path == "/api/v1/diag/events" and not (u.username or u.password or u.query or u.fragment)


def _net_code(exc):
    r = exc.reason if isinstance(exc, urllib.error.URLError) and not isinstance(exc, urllib.error.HTTPError) else exc
    if isinstance(r, socket.gaierror):
        return "DIAGNOSTIC_DNS_ERROR", "dns"
    if isinstance(r, ssl.SSLError) or isinstance(r, ssl.CertificateError):
        return "DIAGNOSTIC_TLS_ERROR", "tls"
    if isinstance(r, (TimeoutError, socket.timeout)):
        return "DIAGNOSTIC_NETWORK_TIMEOUT", "connect" if r is not exc else "read"
    if isinstance(r, (ConnectionError, OSError)):
        return "DIAGNOSTIC_CONNECTION_ERROR", "connect"
    return "DIAGNOSTIC_CONNECTION_ERROR", "unknown"


def post_events(events, key, url=None, timeout=15.0, max_attempts=3, backoff=1.0):   # bounded; the server answers only after the insert committed
    """POST {"events": [...]} once per attempt. Retries ONLY transient failures, with the SAME body (same attempt id and seq): the server's
    conditional write makes a resubmission a duplicate, never a second record. Never retries an authentication or validation answer."""
    url = url or DIAG_URL
    body = json.dumps({"events": events}).encode()
    res = DiagResult(code="DIAGNOSTIC_HTTP_ERROR", attempts=0)
    for i in range(max_attempts):
        res = DiagResult(_post_once(url, body, key, timeout, len(events)), attempts=i + 1)
        if not res.get("transient") or i + 1 == max_attempts:
            return res
        time.sleep(backoff * (2 ** i))
    return res


def _post_once(url, body, key, timeout, n):
    req = urllib.request.Request(url, data=body, method="POST", headers={"Content-Type": "application/json", "Authorization": f"Bearer {key}",
                                                                          "User-Agent": "Enigma-Kaggle-Diagnostic/5.0"})
    t = time.monotonic()
    try:
        with _OPENER.open(req, timeout=timeout) as r:
            status, raw = r.status, r.read(4097)
    except urllib.error.HTTPError as e:
        try:
            raw = e.read(513)
        except Exception:                             # noqa: BLE001
            raw = b""
        finally:
            e.close()
        err = None
        try:
            j = json.loads(raw[:512])
            err = j.get("error") if isinstance(j, dict) and j.get("error") in _SERVER_ERRORS else None
        except ValueError:
            pass
        out = {"http": e.code, "server_error": err, "latency_s": round(time.monotonic() - t, 3)}
        if 300 <= e.code < 400:
            return {**out, "code": "DIAGNOSTIC_HTTP_ERROR", "detail": "redirect_refused"}
        if e.code == 401:
            return {**out, "code": "DIAGNOSTIC_KEY_EXPIRED_OR_REVOKED" if err == "key_expired_or_revoked" else "DIAGNOSTIC_AUTH_REJECTED"}
        if e.code == 429 and err == "quota_exhausted":
            return {**out, "code": "DIAGNOSTIC_QUOTA_EXCEEDED"}
        return {**out, "code": "DIAGNOSTIC_HTTP_ERROR", "transient": e.code in _TRANSIENT_HTTP}
    except (urllib.error.URLError, OSError) as e:      # DNS, TLS, refused, connect timeout (URLError) / read timeout, reset (OSError)
        code, phase = _net_code(e)
        return {"code": code, "phase": phase, "error_type": type(e.reason if isinstance(e, urllib.error.URLError) else e).__name__,
                "transient": code in _TRANSIENT_CODES, "latency_s": round(time.monotonic() - t, 3)}
    out = {"http": status, "latency_s": round(time.monotonic() - t, 3)}
    if status != 200:
        return {**out, "code": "DIAGNOSTIC_HTTP_ERROR"}
    if len(raw) > 4096:
        return {**out, "code": "DIAGNOSTIC_INVALID_RESPONSE", "detail": "oversized"}
    try:
        ack = json.loads(raw)
    except ValueError:
        return {**out, "code": "DIAGNOSTIC_INVALID_RESPONSE", "detail": "not_json"}
    ints = isinstance(ack, dict) and all(isinstance(ack.get(k), int) and not isinstance(ack.get(k), bool) and ack[k] >= 0 for k in ("stored", "duplicates"))
    if not ints:
        return {**out, "code": "DIAGNOSTIC_PERSISTENCE_NOT_ACKNOWLEDGED", "detail": "missing_or_invalid_fields"}
    out.update(stored=ack["stored"], duplicates=ack["duplicates"])
    if ack["stored"] + ack["duplicates"] != n:
        return {**out, "code": "DIAGNOSTIC_PERSISTENCE_NOT_ACKNOWLEDGED", "detail": "count_mismatch"}
    return {**out, "code": "DIAGNOSTIC_PERSISTENCE_VERIFIED"}


def _event(name, rec, fields, seq):
    ev = {"attempt": ATTEMPT, "nb": NOTEBOOK, "seq": seq, "marker": name, "utc": rec["utc"], "t": rec["t"], "experiment": EXPERIMENT,
          "pid": rec["pid"], "pgid": rec["pgid"], "sid": rec["sid"], "mem_available_mib": rec.get("memavailable_mib"),
          "cgroup_current_mib": rec.get("cgroup_current_mib"), "cgroup_max_mib": rec.get("cgroup_max_mib"),
          "oom_kill": rec.get("cgroup_events", {}).get("oom_kill"), "vm_uptime_s": rec.get("vm_uptime_s"),
          "oom_state": "available" if "oom_kill" in rec.get("cgroup_events", {}) else "unavailable", "transport": globals().get("RUNTIME_TRANSPORT")}
    ev.update({k: v for k, v in fields.items() if k in _REMOTE_FIELDS and v is not None})
    return {k: v for k, v in ev.items() if v is not None}


def rmark(name, **fields):
    """mark() locally, then (if remote diagnostics are enabled) one bounded event to the OCI diagnostic route. Returns a DiagResult: truthy only after it
    acknowledged the durable write. A 401 forgets the cached key (fix the secret, then re-run). Nothing secret is ever recorded."""
    rec = mark(name, **fields)
    if not REMOTE_ENABLED:
        return DiagResult(code="REMOTE_DISABLED", marker=name)
    key, cred = get_credential("ENIGMA_DIAG_KEY")
    if key is None:
        mark("remote_skipped", marker_name=name, credential_code=cred["code"])
        print(f"REMOTE NOT-SENT {name}: {cred['code']} -> {cred['message']}", flush=True)
        return DiagResult(code="DIAGNOSTIC_CREDENTIAL_UNAVAILABLE", credential_code=cred["code"], marker=name)
    _DIAG["seq"] += 1
    ev = _event(name, rec, fields, _DIAG["seq"])
    if not _diag_url_ok(DIAG_URL):
        return DiagResult(code="DIAGNOSTIC_HTTP_ERROR", detail="diag_url_not_allowed", marker=name, seq=ev["seq"])
    res = post_events([ev], key)
    res.update(marker=name, seq=ev["seq"])
    if res["code"] in ("DIAGNOSTIC_AUTH_REJECTED", "DIAGNOSTIC_KEY_EXPIRED_OR_REVOKED"):
        _CRED_CACHE.pop("ENIGMA_DIAG_KEY", None)
    _DIAG["last"] = {"event": ev, "result": dict(res)}
    mark("remote_result", **{f"remote_{k}": v for k, v in res.items()})
    print(f"REMOTE {'ACK' if res else 'NOT-ACKNOWLEDGED'} {name} seq={ev['seq']} {res['code']}"
          + ("" if res else f" -> {DIAG_GUIDANCE.get(res['code'], '')}"), flush=True)
    return res


def resubmit_last():
    """Resend the last event UNCHANGED (same attempt id, same seq). A durable record answers duplicates == 1."""
    last = _DIAG.get("last")
    key, _ = get_credential("ENIGMA_DIAG_KEY")
    if not last or key is None:
        return DiagResult(code="DIAGNOSTIC_CREDENTIAL_UNAVAILABLE" if last else "DIAGNOSTIC_PERSISTENCE_NOT_ACKNOWLEDGED")
    res = post_events([last["event"]], key)
    res.update(marker=last["event"]["marker"], seq=last["event"]["seq"])
    mark("remote_resubmit_result", **{f"remote_{k}": v for k, v in res.items()})
    return res


def rcheck(step, name, **fields):
    """A marker that MUST be durable before the next (risky or expensive) operation. Local-only when remote diagnostics are disabled."""
    res = rmark(name, **fields)
    if not REMOTE_ENABLED:
        return res
    if not res:
        verdict(step, "INCONCLUSIVE", f"{name}: {res['code']} -> {DIAG_GUIDANCE.get(res['code'], '')}", remote_code=res["code"])
    return res

## 02-0. Mode and (Mode B only) diagnostic credential + durable OCI acknowledgement. Re-runnable after fixing the secret.

In [ ]:
# 02-0. Experimental mode (set VALIDATION_MODE in the first code cell, before running anything).
#   Mode A "standalone":   strict standalone GPU validation. No network traffic to Enigma, no Kaggle Secret read, local evidence only (the default;
#                          independent of OCI availability).
#   Mode B "oci-assisted": ENIGMA_DIAG_KEY is validated and a durable acknowledgement from the OCI diagnostic route is verified HERE, before any expensive
#                          GPU installation or model start; critical install / server / inference / shutdown markers are persisted there. NOT cloud-independent.
# Neither mode touches Enigma's runtime plane, and neither uses an Enigma runtime token.
REMOTE_ENABLED = VALIDATION_MODE == "oci-assisted"
mark("mode_selected", remote_enabled=REMOTE_ENABLED)
if REMOTE_ENABLED:
    refresh_credentials("ENIGMA_DIAG_KEY")
    check_credential("ENIGMA_DIAG_KEY", "diag_key_preflight")
    first = rmark("diag_first_event", stage="mode", note="oci-assisted GPU validation")
    if not first:
        verdict("remote_ready", "INCONCLUSIVE", f"{first['code']}: {DIAG_GUIDANCE.get(first['code'], '')} (stopping before any GPU work)")
    verdict("remote_ready", "PASS", f"Mode B oci-assisted: durable OCI diagnostic acknowledgement seq={first.get('seq')}")
else:
    verdict("remote_ready", "PASS", "Mode A standalone: no network traffic to Enigma and no secret read; local evidence only")

## 02a. Environment (Python 3.13, 2×T4, Kaggle's torch / triton / transformers)

In [ ]:
require("remote_ready")
import importlib.metadata as md
g = gpu_query()
names = [x.get("name") for x in g.get("gpus", [])]
vers = {p: (md.version(p) if p in {d.metadata["Name"].lower() for d in md.distributions()} else None) for p in ("torch", "triton", "transformers")}
mark("environment", python=platform.python_version(), gpus=names, versions=vers, **g)
ok = names == ["Tesla T4", "Tesla T4"] and vers == {"torch": "2.11.0+cu130", "triton": "3.6.0", "transformers": "5.20.2"}
verdict("environment", "PASS" if ok else "INCONCLUSIVE", f"GPUs {names}, {vers} (required: 2x Tesla T4, torch 2.11.0+cu130, triton 3.6.0, transformers 5.20.2)")

## 02b. Install the immutable build-5 wheel (size + SHA-256 verified first; pydantic-settings conflict must be acknowledged; no new pip-check conflict)

In [ ]:
require("environment")
rcheck("remote_install_begin", "install_begin", stage="install", phase="before")
# Immutable Stokastisk build 5 ONLY. The wheel is downloaded and its exact SIZE and SHA-256 are verified BEFORE pip sees it (the version string 0.1.0
# does not distinguish build tags). Dry-run plan first; the one known replacement must be explicitly acknowledged; then install from the verified file,
# a before/after inventory diff, pip check, and the INSTALLED identity (WHEEL build tag) read without importing anything.
WHEEL_URL = "https://huggingface.co/waqasm86/stokastisk/resolve/main/wheels/stokastisk-0.1.0-5-cp313-cp313-linux_x86_64.whl"
WHEEL_SHA256 = "fe80d7c36fa78cf8a26feb1e00bb5f1f006ca45a027450353603976320ae0b3e"
WHEEL_SIZE = 38057515
KNOWN_REPLACEMENTS = {"pydantic-settings": ["2.16.3", "2.16.2"]}
ACKNOWLEDGE_PYDANTIC_SETTINGS_DOWNGRADE = False      # set True deliberately: build 5 pins pydantic-settings==2.16.2; Kaggle ships 2.16.3 (2.16.3 is a security release)
import hashlib, urllib.request
norm = lambda n: re.sub(r"[-_.]+", "-", n).lower()                                     # noqa: E731
BASE = ("torch", "triton", "transformers", "pydantic", "pydantic-core", "fastapi", "starlette", "numpy")
wheel_path = f"{ATTEMPT_DIR}/stokastisk-0.1.0-5-cp313-cp313-linux_x86_64.whl"
mark("wheel_download_begin", url=WHEEL_URL)
h, n = hashlib.sha256(), 0
with urllib.request.urlopen(WHEEL_URL, timeout=120) as r, open(wheel_path, "wb") as out:
    while True:
        chunk = r.read(1 << 20)
        if not chunk:
            break
        h.update(chunk); n += len(chunk); out.write(chunk)
got = h.hexdigest()
mark("wheel_downloaded", size=n, sha256=got)
if n != WHEEL_SIZE or got != WHEEL_SHA256:
    verdict("wheel_verified", "FAIL", f"NOT the published build 5: size {n} sha256 {got}")
verdict("wheel_verified", "PASS", f"published build 5: {n} bytes, sha256 {got[:16]}...", wheel_sha256=got)


def dependents(name):
    import importlib.metadata as md
    out = []
    for d in md.distributions():
        for req in d.requires or []:
            if norm(re.split(r"[ ;<>=!~\[(]", req.strip(), maxsplit=1)[0]) == name:
                out.append(f"{d.metadata['Name']}=={d.version}: {req}")
    return out


def pip_check():
    r = subprocess.run([sys.executable, "-m", "pip", "check"], capture_output=True, text=True)
    return r.returncode, sorted({l.strip() for l in r.stdout.splitlines() if l.strip() and "no broken requirements" not in l.lower()})


pre_rc, pre_conflicts = pip_check()                    # Kaggle's OWN pre-existing conflicts are recorded, not blamed on build 5
mark("pip_check_before", rc=pre_rc, conflicts=pre_conflicts[:50])
before = inventory()
json.dump(before, open(f"{ATTEMPT_DIR}/packages-before.json", "w"), indent=1, sort_keys=True)
report = f"{ATTEMPT_DIR}/pip-dry-run-report.json"
r = subprocess.run([sys.executable, "-m", "pip", "install", "--dry-run", "--quiet", "--report", report, "--index-url", "https://pypi.org/simple", wheel_path],
                   capture_output=True, text=True)
if r.returncode != 0:
    verdict("install_plan", "FAIL", f"pip dry-run failed rc={r.returncode}", stderr_tail=r.stderr[-1500:])
plan = {norm(i["metadata"]["name"]): i["metadata"]["version"] for i in json.load(open(report))["install"]}
replaced = {k: [before[k], v] for k, v in plan.items() if k in before and before[k] != v}
mark("install_plan", would_install=len(plan), replaced=replaced, dependents={k: dependents(k) for k in replaced})
print("DEPENDENCY_VERSION_PRESERVATION:", "VERIFIED" if not replaced else f"CONFLICT {replaced}", flush=True)
unexpected = {k: v for k, v in replaced.items() if KNOWN_REPLACEMENTS.get(k) != v}
if unexpected:
    verdict("install_plan", "INCONCLUSIVE", f"build 5 would replace unexpected preinstalled packages {unexpected}: not installing", replaced=replaced)
if replaced and not ACKNOWLEDGE_PYDANTIC_SETTINGS_DOWNGRADE:
    verdict("install_plan", "INCONCLUSIVE", f"CONFLICT {replaced}: set ACKNOWLEDGE_PYDANTIC_SETTINGS_DOWNGRADE = True to accept it deliberately", replaced=replaced)
verdict("install_plan", "PASS", f"{len(plan)} to install; replacements {replaced or 'none'}", replaced=replaced)

mark("install_begin")
r = subprocess.run([sys.executable, "-m", "pip", "install", "--index-url", "https://pypi.org/simple", wheel_path], capture_output=True, text=True)
open(f"{ATTEMPT_DIR}/pip-install.log", "w").write(r.stdout[-200000:] + "\n--- stderr ---\n" + r.stderr[-50000:])
after = inventory()
changed = {k: [before[k], after[k]] for k in before if k in after and before[k] != after[k]}
removed = sorted(k for k in before if k not in after)
post_rc, post_conflicts = pip_check()
new_conflicts = sorted(set(post_conflicts) - set(pre_conflicts))
json.dump({"before": pre_conflicts, "after": post_conflicts, "introduced": new_conflicts}, open(f"{ATTEMPT_DIR}/pip-check.json", "w"), indent=1)
import importlib.metadata as md
_dist = md.distribution("stokastisk")
_build = next((l.split(":", 1)[1].strip() for l in (_dist.read_text("WHEEL") or "").splitlines() if l.startswith("Build:")), None)
mark("installed_identity", wheel_build=_build, version=_dist.version, changed=changed, removed=removed, pip_check_rc_before=pre_rc,
     pip_check_rc_after=post_rc, introduced_conflicts=new_conflicts)
base_same = all(before.get(b) == after.get(b) for b in BASE)
ok = (r.returncode == 0 and base_same and changed == replaced and not removed and _build == "5" and _dist.version == "0.1.0"
      and not new_conflicts)                           # PASS never hides a dependency incompatibility that the install introduced
verdict("install", "PASS" if ok else "FAIL", f"rc={r.returncode}, base unchanged={base_same}, changed={changed}, installed build={_build}, "
        f"new pip-check conflicts={new_conflicts or 'none'} (pre-existing: {len(pre_conflicts)})", installed_build=_build, changed=changed,
        introduced_conflicts=new_conflicts)

rmark("install_end", stage="install", wheel_build=_build, wheel_sha256=WHEEL_SHA256)

## 02c. Start the server (loopback `no_proxy` set first)

In [ ]:
require("install")
rcheck("remote_server_begin", "server_start_begin_remote", stage="server", phase="before")
# Start the Stokastisk server with the previously successful settings (TP=1, PP=1, 2048 tokens, gpu_memory_utilization 0.60, Triton attention, fp16, eager).
require("install")
for _k in ("no_proxy", "NO_PROXY"):                 # build 5: loopback HTTP honours http_proxy unless no_proxy lists it; set BEFORE the first request
    _have = [x for x in os.environ.get(_k, "").split(",") if x.strip()]
    os.environ[_k] = ",".join(_have + [h for h in ("127.0.0.1", "localhost", "::1") if h not in _have])
MODEL = "Qwen/Qwen2.5-0.5B-Instruct"
from stokastisk import StokastiskConfig, serving
from stokastisk.serving import ServerProcess
cfg = StokastiskConfig(tensor_parallel_size=1, pipeline_parallel_size=1, max_model_len=2048, gpu_memory_utilization=0.60)
assert (cfg.attention_backend, cfg.dtype, cfg.enforce_eager) == ("TRITON_ATTN", "half", True), "unexpected server defaults"
assert globals().get("server") is None, "a server already exists in this kernel: start a NEW session"
server = ServerProcess(MODEL, cfg, host="127.0.0.1", port=8000, served_model_name=MODEL, log_path=f"{ATTEMPT_DIR}/vllm-server.log")
mark("server_start_begin")
try:
    server.start_and_wait(timeout=900)
except BaseException as e:
    mark("server_start_failed", error=type(e).__name__, log_tail=server.log_tail(20))
    verdict("server_start", "FAIL", f"server did not become healthy: {type(e).__name__}")
procs = group_procs(server.pid)
mark("server_started", server_pid=server.pid, group=procs, **gpu_query())
verdict("server_start", "PASS" if server.health() else "FAIL", f"pid {server.pid}, {len(procs)} process(es) in its group", server_pid=server.pid)

rmark("server_started", stage="server")

## 02d. Real local inference and native vLLM metrics

In [ ]:
# The previously verified local inference test: streamed reply and a rising native vLLM generated-token counter. No Enigma traffic.
require("server_start")
before_m = serving.fetch_metrics(server.base_url)
chunks = list(serving.stream_chat(server.base_url, MODEL, [{"role": "user", "content": "Count from one to ten in words."}], max_tokens=64))
time.sleep(2.5)
after_m = serving.fetch_metrics(server.base_url)
key = next((k for k in after_m if k.startswith("vllm:generation_tokens")), None)
delta = None if key is None else after_m.get(key, 0) - before_m.get(key, 0)
mark("local_inference", chunks=len(chunks), text_chars=len("".join(chunks)), counter=key, generated_delta=delta, **gpu_query())
verdict("local_inference", "PASS" if len(chunks) > 1 and "".join(chunks).strip() and (delta or 0) > 0 else "FAIL",
        f"{len(chunks)} chunks, generated-token counter +{delta}", chunks=len(chunks), generated_delta=delta)

rmark("local_inference_done", stage="inference")

## 02e. Passive server observation: 10 minutes (no remote marker during the window)

In [ ]:
require("local_inference")
def passive_sample():
    procs = group_procs(server.pid)
    states = sorted({p["state"] for p in procs})
    problem = None if server.running and procs and "Z" not in states else f"server_running={server.running} states={states}"
    return {"server_running": server.running, "group_size": len(procs), "states": states, "rss_mib": round(sum(p["rss_mib"] for p in procs), 1),
            "problem": problem, **(gpu_query() if int(time.monotonic()) % 60 < 15 else {})}
observe("server_passive_10min", 600, every=15, sample=passive_sample)
mark("server_log_tail", log_tail=server.log_tail(20))
rmark("server_observed_10min", stage="observe", verdict="PASS")

## 02f. Build-5-safe shutdown

In [ ]:
# Build-5-safe shutdown (no connector needed): build-5 ServerProcess.stop() is only safe while the server's leader process is unreaped. If the leader
# already exited, stop() is NOT called (it could SIGKILL a reused process group); the exit is reported instead.
p = getattr(server, "_proc", None)
if p is not None and p.poll() is None:
    rc = server.stop()
    mark("server_stopped", rc=rc, report=server.shutdown_report, **gpu_query())
    verdict("server_shutdown", "PASS" if server.shutdown_report.get("group_gone") else "FAIL", f"rc={rc} {server.shutdown_report}")
else:
    mark("server_leader_already_exited", rc=None if p is None else p.returncode, log_tail=server.log_tail(20))
    verdict("server_shutdown", "FAIL", "the server process had already exited before shutdown; build-5 stop() deliberately not called")
rmark("final", verdict=RESULTS["server_shutdown"]["status"])

## Evidence export (local, redacted)

In [ ]:
# Evidence export (run it even after a FAIL / INCONCLUSIVE stop). Review the archive before sharing it; it contains no credential.
mark("export_begin", results={k: v["status"] for k, v in RESULTS.items()})
archive = export_evidence()
print(json.dumps({k: {"status": v["status"], "reason": v["reason"]} for k, v in RESULTS.items()}, indent=1), flush=True)